In [1]:
from pathlib import Path
import contextlib, io, json, sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import step3_collect_urls as s3

RUN_REFRESH = False  # True: tải lại toàn bộ 104 task; False: dùng cache

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    s3.main(refresh=RUN_REFRESH)

DATA = PROJECT_ROOT / "data" / "step3"
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
summary = pd.read_csv(DATA / "summary_by_label.csv", encoding="utf-8-sig")
candidates = pd.read_csv(DATA / "url_candidates.csv", encoding="utf-8-sig")
duplicates = pd.read_csv(DATA / "duplicates_review.csv", encoding="utf-8-sig")
gaps = pd.read_csv(DATA / "gaps.csv", encoding="utf-8-sig")

print("Đã sinh danh sách URL ứng viên.")
print({k: manifest[k] for k in ["tasks_total", "cache_hits", "fetched",
                                "network_requests", "article_content_requests",
                                "selected", "gap_total"]})

Đã sinh danh sách URL ứng viên.
{'tasks_total': 104, 'cache_hits': 104, 'fetched': 0, 'network_requests': 0, 'article_content_requests': 0, 'selected': 6000, 'gap_total': 0}


In [2]:
assert len(candidates) == 6000
assert candidates.Candidate_ID.nunique() == 6000
assert candidates.URL_Normalized.nunique() == 6000
assert set(candidates.Label) == {"Kinh tế", "Thế giới", "Thể thao", "Pháp luật",
                                 "Giáo dục", "Sức khỏe", "Công nghệ", "Du lịch"}
assert set(candidates.Source) == {"VnExpress", "TuoiTre", "ThanhNien", "DanTri", "VietnamNet"}
assert set(candidates.Collection_Method) <= {"rss", "category"}
assert candidates.Original_Category.notna().all()
assert gaps.Gap.sum() == 0
summary

,Label,VnExpress,TuoiTre,ThanhNien,DanTri,VietnamNet,Total,Target,Gap
0,Kinh tế,150,50,50,100,400,750,750,0
1,Thế giới,150,50,50,100,400,750,750,0
2,Thể thao,150,50,50,100,400,750,750,0
3,Pháp luật,150,50,50,100,400,750,750,0
4,Giáo dục,150,50,50,100,400,750,750,0
5,Sức khỏe,150,50,50,100,400,750,750,0
6,Công nghệ,150,50,50,100,400,750,750,0
7,Du lịch,150,50,50,100,400,750,750,0


In [3]:
by_source = candidates.groupby("Source").size().rename("Selected")
by_method = candidates.groupby(["Source", "Collection_Method"]).size().unstack(fill_value=0)
display(pd.concat([by_source, by_method], axis=1).fillna(0).astype(int))

assert manifest["article_content_requests"] == 0
assert manifest["selected"] == 6000
assert manifest["gap_total"] == 0

,Selected,category,rss
Source,,,
DanTri,800,1,799
ThanhNien,400,0,400
TuoiTre,400,4,396
VietnamNet,3200,400,2800
VnExpress,1200,1200,0


## 3. Cache và khả năng chạy lại

Mỗi trang danh mục/RSS có một file cache JSON. Lần chạy lại cuối cùng đọc **104/104 task từ cache**, không gửi request mạng và vẫn tạo lại đúng 6.000 dòng.

In [4]:
cache_info = pd.DataFrame([{
    "Tasks": manifest["tasks_total"],
    "Cache hits": manifest["cache_hits"],
    "Fetched": manifest["fetched"],
    "Network requests": manifest["network_requests"],
    "Article content requests": manifest["article_content_requests"],
    "Errors": manifest["errors"],
}])
cache_info

,Tasks,Cache hits,Fetched,Network requests,Article content requests,Errors
0,104,104,0,0,0,0


In [5]:
print("Số trường hợp URL trùng:", manifest["duplicate_url_cases"])
print("Số nhóm tiêu đề trùng:", manifest["potential_title_duplicate_cases"])
print("Số bản ghi bị gắn cờ:", int((candidates.Potential_Duplicate == "yes").sum()))
print("Phân bố:")
print(duplicates.Duplicate_Type.value_counts().to_string())

duplicates[["Duplicate_Type", "Group_ID", "Primary_ID", "Duplicate_ID",
            "Source_Primary", "Source_Duplicate", "Label_Primary", "Label_Duplicate",
            "Title_Primary", "Action"]].head(20)

Số trường hợp URL trùng: 20
Số nhóm tiêu đề trùng: 6
Số bản ghi bị gắn cờ: 12
Phân bố:
Duplicate_Type
same_url      20
same_title     6


,Duplicate_Type,Group_ID,Primary_ID,Duplicate_ID,Source_Primary,Source_Duplicate,Label_Primary,Label_Duplicate,Title_Primary,Action
0,same_url,URL00195,P000195,P009169,TuoiTre,TuoiTre,Kinh tế,Du lịch,'Kinh tế bạc': Ngành du lịch làm gì để thu hút...,merged_keep_primary
1,same_url,URL00218,P000218,P001569,TuoiTre,TuoiTre,Kinh tế,Thế giới,"Vì sao cần thanh tra phòng, chống rửa tiền với...",merged_keep_primary
2,same_url,URL01662,P001663,P005758,DanTri,DanTri,Thế giới,Giáo dục,Hơn 550.000 học sinh Hàn Quốc thi đại học giữa...,merged_keep_primary
3,same_url,URL02809,P002810,P002814,VnExpress,VnExpress,Thể thao,Thể thao,Klopp: 'FIFA thời Infantino còn tệ hơn thời Bl...,merged_keep_primary
4,same_url,URL02810,P002811,P002815,VnExpress,VnExpress,Thể thao,Thể thao,Cựu trợ lý của HLV Park Hang-seo tái xuất V-Le...,merged_keep_primary
5,same_url,URL02811,P002812,P002816,VnExpress,VnExpress,Thể thao,Thể thao,HLV Moriyasu và hành trình không trọn vẹn cùng...,merged_keep_primary
6,same_url,URL02812,P002813,P002817,VnExpress,VnExpress,Thể thao,Thể thao,Bí ẩn viên kim cương 300.000 USD biến mất sau ...,merged_keep_primary
7,same_url,URL05650,P005655,P009172,TuoiTre,TuoiTre,Giáo dục,Du lịch,Phú Quốc cần hơn 14.000 nhân sự phục vụ APEC 2027,merged_keep_primary
8,same_url,URL05665,P005670,P007030,TuoiTre,TuoiTre,Giáo dục,Sức khỏe,Vụ 22 học sinh nghi ngộ độc do uống nước ngọt:...,merged_keep_primary
9,same_url,URL08571,P008578,P008654,VietnamNet,VietnamNet,Công nghệ,Công nghệ,CMC Telecom là doanh nghiệp Việt Nam duy nhất ...,merged_keep_primary


In [6]:
print("Thiếu tiêu đề:", int(candidates.Title.isna().sum()))
print("Thiếu ngày đăng:", int(candidates.Published_Date.isna().sum()))
print("Có ngày đăng:", int(candidates.Published_Date.notna().sum()))
assert candidates.Title.notna().all()
assert candidates.Published_Date.isna().sum() < len(candidates)

candidates.sample(8, random_state=42)[
    ["Candidate_ID", "Source", "Label", "Original_Category",
     "Collection_Method", "Title", "Published_Date"]
]

Thiếu tiêu đề: 0
Thiếu ngày đăng: 1145
Có ngày đăng: 4855


,Candidate_ID,Source,Label,Original_Category,Collection_Method,Title,Published_Date
1782,C001783,DanTri,Thể thao,Thể thao,rss,Zverev lần thứ 5 giành vé vào bán kết ATP Finals,2026-11-18T08:32:33+07:00
3917,C003918,TuoiTre,Sức khỏe,Sức khỏe,rss,Miền Nam sáng sớm và trưa nhiệt độ chênh nhau ...,2026-11-19T09:19:00+07:00
221,C000222,ThanhNien,Kinh tế,Kinh tế,rss,Chính thức đề xuất giảm phạt gần 12.700 tỉ đồn...,2026-11-19T17:39:00+07:00
2135,C002136,VietnamNet,Thể thao,Thể thao,rss,Luis Enrique yêu cầu PSG mua Kobbie Mainoo,2026-11-06T17:40:24+07:00
5224,C005225,VietnamNet,Công nghệ,Công nghệ,category,"Tên lửa Steelfin hai tầng, khắc tinh của tên l...",NaN
1168,C001169,VietnamNet,Thế giới,Thế giới,rss,Hai máy bay Boeing va quệt nhau tại sân bay Mỹ,2026-11-15T13:45:00+07:00
879,C000880,VnExpress,Thế giới,Thế giới,category,Chiếc router khiến không phận Mỹ tê liệt,NaN
156,C000157,TuoiTre,Kinh tế,Kinh doanh,rss,30 năm chứng khoán: Chuyện hậu trường ít biết ...,2026-11-20T08:15:00+07:00
